In [ ]:
!pip install -U torch


In [ ]:
from src.database_builder import get_db_connection
from src.data_fetch import fetch_raw_from_db
from dotenv import load_dotenv

In [ ]:
HF_TOKEN = load_dotenv('HF_TOKEN')

In [ ]:
conn = get_db_connection()
raw_data = fetch_raw_from_db(conn=conn)

In [46]:
df = raw_data['sec']
df['content'] = df['content'].astype(str)


In [47]:
out = (
    df.groupby(["filing_date", "item_type"], as_index=False)
      .agg(content=("content", list))
)

In [29]:
out


,filing_date,item_type,content
0,2018-01-31,MDA,"[Economic Conditions, Challenges, and Risks\nT..."
1,2018-01-31,RiskFactorsUpdate,[ITEM 1A. RISK FACTORS Our operations and fina...
2,2018-04-26,MDA,"[Economic Conditions, Challenges, and Risks\nT..."
3,2018-04-26,RiskFactorsUpdate,[ITEM 1A. RISK FACTORS Our operations and fina...
4,2018-08-03,MDA,"[Commercial cloud revenue, which primarily com..."
...,...,...,...
61,2025-07-30,RiskFactors,[ITEM 1A. RISK FACTORS Our operations and fina...
62,2025-10-29,MDA,"[Economic Conditions, Challenges, and Risks\nT..."
63,2025-10-29,RiskFactorsUpdate,[ITEM 1A. RISK FACTORS Our operations and fina...
64,2026-01-28,MDA,"[Economic Conditions, Challenges, and Risks\nT..."


In [ ]:
out.loc(('filing_date=="")& ('item__type'=='mda ),:)

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (3665162301.py, line 1)

In [48]:
mask_q1 = (out['filing_date'] == '2018-01-31') & (out['item_type'] == 'MDA')
q1_mda = out[mask_q1]['content']
mask_q2 = (out['filing_date'] == '2018-04-26') & (out['item_type'] == 'MDA')
q2_mda = out[mask_q2]['content']

AttributeError: 'list' object has no attribute 'split'

In [14]:
from sentence_transformers import SentenceTransformer, util
from transformers import BertTokenizer, BertModel

import torch

#Intialize model (check logged as sentance transformer or classification on HF)
model_name = 'yiyanghkust/finbert-tone'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)







Loading weights: 100%|██████████| 199/199 [00:00<00:00, 25260.01it/s]
[transformers] BertModel LOAD REPORT from: yiyanghkust/finbert-tone
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
def encode_finance_text(text_list):
    # Tokenize and move to tensors
    inputs = tokenizer(text_list, padding=True, truncation=True, return_tensors="pt")
    
    with torch.no_grad():
        outputs = model(**inputs)
        # We take the 'last_hidden_state' (the 768-dim vectors for each word)
        last_hidden_state = outputs.last_hidden_state
        
        # MEAN POOLING: Average the word vectors to get one sentence vector
        # This is exactly what SentenceTransformers does under the hood
        embeddings = torch.mean(last_hidden_state, dim=1)
        
    return embeddings

In [59]:
# 2. Simulate Microsoft 10-K Risk Sections (Quarter-over-Quarter)
# Q3: Original text
q3_risk_section = [
    "We face intense competition in all our markets.", # Chunk 0
    "Our cloud computing business depends on data center uptime.", # Chunk 1
    "Fluctuations in foreign exchange rates may affect our results." # Chunk 2
]

# Q4: Updated text (We added a new risk and shifted the old ones)
q4_risk_section = [
    "We face intense competition in all our markets.", # Same as Q3
    "NEW: Emerging AI regulations in Europe may impact our cloud margins.", # BRAND NEW
    "Our cloud computing business depends on data center uptime.", # Shifted from ID 1 to ID 2
    "Exchange rate fluctuations can impact our international revenue." # Paraphrased version of Q3 Chunk 2
]

# 3. Step A: Encode all chunks into Vectors
q3_embeddings = encode_finance_text(q1_mda[0] )
q4_embeddings = encode_finance_text(q2_mda[0])



KeyError: 0

In [18]:
q3_embeddings.shape

torch.Size([3, 768])

In [19]:
# 4. Step B: Semantic Alignment (The "Many-to-One" Search)
print(f"{'New Chunk Text':<60} | {'Best Match Score':<15} | {'Status'}")
print("-" * 90)

drill_down_features = []

for i, q4_vec in enumerate(q4_embeddings):
    # Calculate cosine similarity between THIS new chunk and ALL old chunks
    # This is the "Search" part that ignores position shifts
    cosine_scores = util.cos_sim(q4_vec, q3_embeddings)
    
    # Find the best score and the index of that match
    best_score = torch.max(cosine_scores).item()
    
    status = "REUSE" if best_score > 0.95 else "PARAPHRASE" if best_score > 0.8 else "NEW SIGNAL"
    
    print(f"{q4_risk_section[i][:58]:<60} | {best_score:<15.4f} | {status}")
    drill_down_features.append(best_score)

# 5. Step C: Generate Features for LightGBM
max_dev = min(drill_down_features) # The "Maximum Surprise"
avg_dev = sum(drill_down_features) / len(drill_down_features)

print(f"\n--- Model Features for LightGBM ---")
print(f"Minimum Similarity (High Deviation): {max_dev:.4f}")
print(f"Average Document Similarity: {avg_dev:.4f}")

New Chunk Text                                               | Best Match Score | Status
------------------------------------------------------------------------------------------
We face intense competition in all our markets.              | 0.9913          | REUSE
NEW: Emerging AI regulations in Europe may impact our clou   | 0.6625          | NEW SIGNAL
Our cloud computing business depends on data center uptime   | 0.9948          | REUSE
Exchange rate fluctuations can impact our international re   | 0.8641          | PARAPHRASE

--- Model Features for LightGBM ---
Minimum Similarity (High Deviation): 0.6625
Average Document Similarity: 0.8782
